In [ ]:
# --- 6. Solve the Model ---
print("=== Starting Optimization Process ===")
print(f"Problem: {prob.name}")
print(f"Objective: {prob.sense} (1=Minimize, -1=Maximize)")
print(f"Variables: {len(prob.variables())}")
print(f"Constraints: {len(prob.constraints)}")

# Add detailed logging
print("\n=== Problem Statistics ===")
print(f"Process requirements:")
for p, req in process_requirements.items():
    print(f"  {p}: {req:,.1f} minutes/week")

print(f"\nMachine types available: {len(MACHINE_NAMES)}")
print(f"Effective working time per machine: {EFFECTIVE_MINS_PER_WEEK:,.1f} minutes/week")

# Solve with verbose output
print("\n=== Solving with CBC Solver ===")
solver_to_use = pulp.PULP_CBC_CMD(msg=True, timeLimit=300, fracGap=0.01)  # 5 min limit, 1% gap
print("Solver parameters: timeLimit=300s, fracGap=0.01")

import time
start_time = time.time()
prob.solve(solver_to_use)
solve_time = time.time() - start_time

print(f"\n=== Solve Complete in {solve_time:.2f} seconds ===")
print(f"Solver status code: {prob.status}")
print(f"Status description: {pulp.LpStatus[prob.status]}")

# Additional diagnostics
if prob.status != pulp.LpStatusOptimal:
    print("\n=== DIAGNOSTIC INFORMATION ===")
    print("Problem may be infeasible or unbounded")
    
    # Check if any constraints might be problematic
    print("\nChecking constraint feasibility...")
    total_min_demand = sum(process_requirements.values())
    print(f"Total minimum demand: {total_min_demand:,.1f} minutes/week")
    
    # Check if we have enough total capacity theoretically
    total_theoretical_capacity = len(MACHINE_NAMES) * EFFECTIVE_MINS_PER_WEEK
    print(f"Max theoretical capacity (if bought all machines): {total_theoretical_capacity:,.1f} minutes/week")
    
    if total_min_demand > total_theoretical_capacity:
        print("❌ INFEASIBLE: Total demand exceeds maximum possible capacity!")
    else:
        print("✅ Demand vs capacity check passed")
        
    # Check for constraint conflicts
    print("\nConstraint summary:")
    for name, constraint in prob.constraints.items():
        print(f"  {name}: {constraint}")
        
print(f"\n=== Solution Analysis ===")
if prob.status == pulp.LpStatusOptimal:
    print("✅ OPTIMAL SOLUTION FOUND")
elif prob.status == pulp.LpStatusInfeasible:
    print("❌ PROBLEM IS INFEASIBLE")
elif prob.status == pulp.LpStatusUnbounded:
    print("⚠️ PROBLEM IS UNBOUNDED")
elif prob.status == pulp.LpStatusNotSolved:
    print("⏱️ SOLVER TIMED OUT OR NOT SOLVED")
else:
    print(f"❓ UNKNOWN STATUS: {prob.status}")

# --- 7. Print the Results ---
print(f"--- Solve Complete ---")

# We compare prob.status to pulp.LpStatusOptimal (which is 1)
print(f"Status: {pulp.LpStatus[prob.status]}")

if prob.status == pulp.LpStatusOptimal: # <-- This is the fix
    
    # Print Total Cost
    total_cost_f1 = pulp.value(prob.objective)
    print(f"\n--- Optimal Equipment Plan for 'Fractal Center F1' ---")
    print(f"Minimum Equipment Investment for a Single Center (F1): ${total_cost_f1:,.2f}")
    
    # Calculate Total Factory Investment
    total_factory_cost = total_cost_f1 * 4 # f=4
    print(f"Total Factory Equipment Investment (F1, F2, F3, F4): ${total_factory_cost:,.2f}")

    print("\n--- Optimal Machine Mix for F1 Center ---")
    total_machines_f1 = 0
    machine_mix_f1 = {}
    for m in MACHINE_NAMES:
        if n_machines[m].varValue is not None:
            count = int(round(n_machines[m].varValue))
            if count > 0:
                print(f"  {m}: {count} units")
                total_machines_f1 += count
                machine_mix_f1[m] = count
                
    print(f"  ---------------------")
    print(f"  Total Machines in F1 Center: {total_machines_f1} units")
    print(f"  Total Factory Machines (f=4): {total_machines_f1 * 4} units")

    print("\n--- Time Allocations for F1 Center (showing > 0) ---")
    time_allocations_f1 = {}
    for (m, p) in time_allocations:
        if time_allocations[(m, p)].varValue is not None and time_allocations[(m, p)].varValue > 0.1:
            alloc_time = time_allocations[(m, p)].varValue
            print(f"  {m} -> {p}: {alloc_time:,.1f} minutes")
            if (m,p) not in time_allocations_f1:
                time_allocations_f1[(m,p)] = 0
            time_allocations_f1[(m,p)] += alloc_time

else:
    print("Optimal solution not found. Please check the solver logs.")

In [21]:
import pandas as pd
import numpy as np
import io

# --- 1. 基础数据准备 ---

# (A) F1 中心的每周零件需求 (总需求 / 4)
# (数据来自您在 Task 1 中计算的 Target_Weekly_Capacity_Final)
part_demand_f1 = {
    'P1': 30607.0 / 4, 'P2': 15932.0 / 4, 'P3': 8294.0 / 4, 'P4': 13511.0 / 4,
    'P5': 10490.0 / 4, 'P6': 5836.0 / 4, 'P7': 20660.0 / 4, 'P8': 6043.0 / 4,
    'P9': 13404.0 / 4, 'P10': 6633.0 / 4, 'P11': 11672.0 / 4, 'P12': 14340.0 / 4,
    'P13': 7170.0 / 4, 'P14': 26755.0 / 4, 'P15': 3022.0 / 4, 'P16': 18066.0 / 4,
    'P17': 7170.0 / 4, 'P18': 17253.0 / 4, 'P19': 29273.0 / 4, 'P20': 21724.0 / 4
}

# (B) F1 中心的设备清单 (来自您的优化结果图片)
machine_mix_f1 = {
    'A': 4, 'B': 13, 'C': 3, 'ABCD': 19, 'EF': 15, 'FG': 7,
    'H': 13, 'I': 12, 'J': 19, 'KL': 3, 'LM': 32
}

# (C) 零件工艺路线 (来自 'Parts Specs.csv')
# 为了运行，我们直接使用 'Parts Specs.csv' 的内容
parts_specs_csv = """
Part,Process_1,Process_2,Process_3,Process_4,Process_5,Process_6,Process_7,Process_8
P1,A,B,C,D,I,J,,
P2,A,C,D,H,J,,,
P3,B,C,D,I,J,,,
P4,A,B,D,G,H,,,
P5,B,C,D,I,,,,
P6,A,B,C,D,H,I,J,
P7,C,D,E,F,I,J,,
P8,E,H,I,J,,,,
P9,E,F,G,I,J,,,
P10,E,F,I,J,,,,
P11,E,G,I,,,,,
P12,E,F,G,I,J,,,
P13,E,F,G,H,I,,,
P14,E,F,G,H,,,,
P15,E,F,G,H,J,,,
P16,F,H,I,J,,,,
P17,K,L,M,,,,,
P18,K,L,M,,,,,
P19,L,M,,,,,,
P20,K,L,M,,,,,
"""

# (D) 设备面积 (来自 'Basic Layouts.csv')
basic_layouts_csv = """
Machine,Length,Width,Area
A,15,10,150
B,15,15,225
C,15,15,225
D,30,20,600
E,20,15,300
F,20,15,300
G,20,15,300
H,40,20,800
I,20,20,400
J,20,20,400
K,10,10,100
L,10,10,100
M,10,10,100
AB,20,15,300
AC,20,15,300
CD,30,20,600
ABC,30,15,450
ABCD,30,20,600
EF,10,10,100
EG,20,15,300
FG,10,10,100
EFG,20,15,300
IJ,25,20,500
KL,10,10,100
KM,25,10,250
LM,10,10,100
KLM,10,10,100
"""

# --- 2. 定义 F1 中心的6个“布局块” ---
# 基于优化结果，我们将工艺 (A-M) 映射到它们所属的机器组
process_to_group_map = {
    'A': 'ABCD_Group', 'B': 'ABCD_Group', 'C': 'ABCD_Group', 'D': 'ABCD_Group',
    'E': 'EFG_Group', 'F': 'EFG_Group', 'G': 'EFG_Group',
    'H': 'H_Group',
    'I': 'I_Group',
    'J': 'J_Group',
    'K': 'KLM_Group', 'L': 'KLM_Group', 'M': 'KLM_Group'
}
groups = sorted(list(set(process_to_group_map.values())))
# groups = ['ABCD_Group', 'EFG_Group', 'H_Group', 'I_Group', 'J_Group', 'KLM_Group']

# --- 3. 计算每个“块”的面积 (交付成果 3 的基础) ---
print("--- 阶段 2.1: 'F1 中心' 布局块面积计算 (交付成果 3 基础) ---")
layout_df = pd.read_csv(io.StringIO(basic_layouts_csv))
layout_df = layout_df.set_index('Machine')
# 将 'Area' 转换为数值
layout_df['Area'] = pd.to_numeric(layout_df['Area'])

# 定义哪些机器属于哪个块 (基于 F1 machine_mix)
group_machines = {
    'ABCD_Group': ['A', 'B', 'C', 'ABCD'],
    'EFG_Group': ['EF', 'FG'], # E, F, G 专机未被购买
    'H_Group': ['H'],
    'I_Group': ['I'],
    'J_Group': ['J'],
    'KLM_Group': ['KL', 'LM'] # K, L, M 专机未被购买
}

block_areas = {}
for group, machines_in_group in group_machines.items():
    total_area = 0
    for machine_name, count in machine_mix_f1.items():
        if machine_name in machines_in_group:
            total_area += layout_df.loc[machine_name, 'Area'] * count
    block_areas[group] = total_area

# 打印面积结果
area_df = pd.DataFrame.from_dict(block_areas, orient='index', columns=['Total_Area_sqft'])
print(area_df.to_markdown(floatfmt=",.0f"))
print("\n")


# --- 4. 计算“块”之间的流量矩阵 (交付成果 4) ---
print("--- 阶段 2.2: 'F1 中心' 内部流量矩阵 (交付成果 4) ---")
specs_df = pd.read_csv(io.StringIO(parts_specs_csv))
specs_df = specs_df.set_index('Part')

# 初始化一个空的 From-To 矩阵
flow_matrix = pd.DataFrame(0, index=groups, columns=groups)

# 遍历所有20个零件
for part, routing in specs_df.iterrows():
    demand = part_demand_f1.get(part, 0)
    if demand == 0:
        continue
        
    # 获取工艺步骤列表，并去除空值
    steps = [step for step in routing if pd.notna(step)]
    
    # 遍历工艺路线 (例如 A -> B, B -> C, ...)
    for i in range(len(steps) - 1):
        process_from = steps[i]
        process_to = steps[i+1]
        
        # 映射到对应的“块”
        group_from = process_to_group_map[process_from]
        group_to = process_to_group_map[process_to]
        
        # 仅当流量在 *不同* 块之间移动时，我们才记录它
        # (块内部的流量对于块布局不重要)
        if group_from != group_to:
            flow_matrix.loc[group_from, group_to] += demand

# 打印流量矩阵结果
print("流量单位: 周转次数/周 (Trips/Week)")
# 四舍五入到整数，便于阅读
flow_matrix = flow_matrix.round(0).astype(int)
print(flow_matrix.to_markdown())

--- 阶段 2.1: 'F1 中心' 布局块面积计算 (交付成果 3 基础) ---
|            |   Total_Area_sqft |
|:-----------|------------------:|
| ABCD_Group |            15,600 |
| EFG_Group  |             2,200 |
| H_Group    |            10,400 |
| I_Group    |             4,800 |
| J_Group    |             7,600 |
| KLM_Group  |             3,500 |


--- 阶段 2.2: 'F1 中心' 内部流量矩阵 (交付成果 4) ---
流量单位: 周转次数/周 (Trips/Week)
|            |   ABCD_Group |   EFG_Group |   H_Group |   I_Group |   J_Group |   KLM_Group |
|:-----------|-------------:|------------:|----------:|----------:|----------:|------------:|
| ABCD_Group |            0 |        8543 |      5442 |     12348 |         0 |           0 |
| EFG_Group  |            0 |           0 |     18642 |     16677 |         0 |           0 |
| H_Group    |            0 |           0 |         0 |      9279 |      4738 |           0 |
| I_Group    |            0 |           0 |         0 |         0 |     30971 |           0 |
| J_Group    |            0 |           0 | 

C:\Users\11573\AppData\Local\Temp\ipykernel_26056\2022064120.py:154: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '7651.75' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  flow_matrix.loc[group_from, group_to] += demand
C:\Users\11573\AppData\Local\Temp\ipykernel_26056\2022064120.py:154: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '7651.75' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  flow_matrix.loc[group_from, group_to] += demand
C:\Users\11573\AppData\Local\Temp\ipykernel_26056\2022064120.py:154: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '3377.75' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  flow_matrix.loc[